# 00 · Python 快速复健（为《Kalman and Bayesian Filters in Python》做准备）

目标：**1–2 小时**内把读这本书需要的 Python 捡回来。不求全，只求够用。

你已经会 R、MATLAB、Julia，所以每节都会标出 **和 R/MATLAB 的区别**——这些是最容易踩的坑。

**用法**：选中代码格，按 `Shift + Enter` 运行。每节末尾有 ✏️ 小练习，先自己写，再看下面的参考答案。

## 1. 变量、类型、打印

> ⚠️ 与 R/MATLAB 的区别：赋值用 `=`（不是 `<-`）；行尾不需要 `;`；**缩进是语法的一部分**。

In [ ]:
x = 3          # int
y = 2.5        # float
name = "kalman"  # str
ok = True      # bool（注意大写 T）

print(x + y)
print(type(x), type(y))
print(f"{name} 的长度是 {len(name)}，x*y = {x*y:.2f}")   # f-string，最常用的格式化方式

In [ ]:
# 整数除法 vs 普通除法
print(7 / 2)    # 3.5
print(7 // 2)   # 3   （向下取整）
print(7 % 2)    # 1   （余数）
print(2 ** 10)  # 1024 （幂，不是 ^ ！）

## 2. 列表（list）与切片

> ⚠️ **最大的坑**：Python 从 **0** 开始数（R/MATLAB 从 1）；切片 `a[i:j]` **包含 i、不包含 j**。

In [ ]:
a = [10, 20, 30, 40, 50]
print(a[0])      # 10  —— 第一个
print(a[-1])     # 50  —— 最后一个
print(a[1:3])    # [20, 30] —— 下标 1、2，不含 3
print(a[:2])     # [10, 20]
print(a[2:])     # [30, 40, 50]
print(len(a))

a.append(60)     # 在末尾追加（书里大量用这个来存每一步的估计值）
print(a)

In [ ]:
# 列表里的 + 是“拼接”，不是逐元素相加！
print([1, 2] + [3, 4])    # [1, 2, 3, 4]
print([1, 2] * 2)         # [1, 2, 1, 2]
# 想要逐元素运算 -> 用 numpy（第 6 节）

## 3. 循环

> ⚠️ `range(n)` 产生 `0, 1, ..., n-1`。循环体靠**缩进**界定，没有 `end` / `}`。

In [ ]:
for i in range(3):
    print("step", i)

measurements = [158.0, 164.2, 160.3]
for i, z in enumerate(measurements):      # 同时拿到下标和值
    print(f"第 {i} 次测量: {z}")

preds = [160.0, 161.0, 162.0]
for p, z in zip(preds, measurements):     # 两个列表并排走
    print(f"预测 {p}, 测量 {z}, 残差 {z - p:+.1f}")

In [ ]:
# 列表推导式：一行生成列表（书里常见）
squares = [k**2 for k in range(5)]
print(squares)

## 4. 条件判断

In [ ]:
residual = 4.2
if abs(residual) > 5:
    print("残差太大，可能是异常值")
elif abs(residual) > 2:
    print("残差偏大")
else:
    print("正常")
# 逻辑运算用英文单词：and / or / not（不是 && || !）

## 5. 函数

> 书里几乎每个滤波步骤都会写成函数，比如 `predict(...)`、`update(...)`。

In [ ]:
def weighted_mean(a, b, w=0.5):
    # w 是默认参数：不传就用 0.5
    return w * a + (1 - w) * b

print(weighted_mean(160, 170))          # 165.0
print(weighted_mean(160, 170, w=0.8))   # 162.0

def stats(xs):
    m = sum(xs) / len(xs)
    var = sum((x - m) ** 2 for x in xs) / len(xs)
    return m, var          # 一次返回多个值（其实是一个 tuple）

mean, var = stats([1, 2, 3, 4])
print(mean, var)

✏️ **练习 5**：写一个函数 `clip(x, lo, hi)`，把 `x` 限制在 `[lo, hi]` 范围内。

In [ ]:
def clip(x, lo, hi):
    # TODO: 你的代码
    pass

# print(clip(5, 0, 3))   # 期望 3
# print(clip(-1, 0, 3))  # 期望 0

<details><summary>参考答案（点开）</summary>

```python
def clip(x, lo, hi):
    if x < lo:
        return lo
    if x > hi:
        return hi
    return x
```
</details>

## 6. NumPy —— 这本书的主力

> NumPy 的 array 和 MATLAB 的矩阵/R 的向量很像：**逐元素运算**。
> ⚠️ 区别：`*` 是**逐元素乘**；矩阵乘法用 `@`；转置用 `.T`。

In [ ]:
import numpy as np

v = np.array([1.0, 2.0, 3.0])
print(v * 2)          # [2. 4. 6.]  逐元素
print(v + v)          # [2. 4. 6.]
print(v.mean(), v.std(), v.var())
print(v.shape)        # (3,)

In [ ]:
F = np.array([[1.0, 1.0],
              [0.0, 1.0]])   # 2x2 矩阵（后面卡尔曼滤波的“状态转移矩阵”就长这样）
x = np.array([[10.0],
              [ 2.0]])       # 2x1 列向量：位置=10，速度=2

print(F @ x)          # 矩阵乘法 -> [[12.], [2.]]
print(F * F)          # ⚠️ 逐元素乘，不是矩阵乘！
print(F.T)            # 转置
print(np.linalg.inv(F))   # 逆矩阵
print(F.shape, x.shape)

In [ ]:
# 随机数：模拟“带噪声的传感器”
rng = np.random.default_rng(seed=42)    # 固定种子 -> 结果可复现
true_weight = 160.0
noise = rng.normal(loc=0.0, scale=3.0, size=10)   # 均值 0、标准差 3
z = true_weight + noise
print(np.round(z, 1))

# 书里常见的旧写法（效果类似）：
# np.random.seed(42); np.random.randn(10)

In [ ]:
# 常用构造函数
print(np.zeros(3), np.ones(3))
print(np.eye(2))                 # 单位阵
print(np.arange(0, 1, 0.25))     # [0. 0.25 0.5 0.75]（不含终点）
print(np.linspace(0, 1, 5))      # [0. 0.25 0.5 0.75 1.]（含终点）

✏️ **练习 6**：用 `rng.normal` 生成 1000 个均值 5、标准差 2 的样本，验证样本均值≈5、方差≈4。

In [ ]:
# TODO

<details><summary>参考答案</summary>

```python
s = rng.normal(5, 2, size=1000)
print(s.mean(), s.var())
```
</details>

## 7. Matplotlib 画图

In [ ]:
import matplotlib.pyplot as plt

t = np.arange(10)
plt.figure(figsize=(7, 3.5))
plt.scatter(t, z, color="tab:red", label="测量值 z（含噪声）")
plt.axhline(true_weight, color="k", ls="--", label="真实值")
plt.xlabel("时间步")
plt.ylabel("体重")
plt.legend()
plt.show()

（如果中文标签显示成方块，是字体问题，不影响学习；可以把 label 改成英文。）

## 8. 热身：自己写一个最朴素的“滤波器”

这一节直接衔接书的**第 1 章（g-h filter）**。

场景：每天称一次体重，秤有噪声（±3）。我们相信体重变化不会太剧烈。
思路——每一步做两件事：

1. **预测**：明天的体重 ≈ 今天的估计值
2. **更新**：拿到新测量 `z` 后，把估计往测量方向“挪一点”：
   `estimate = prediction + g * (z - prediction)`，其中 `0 < g < 1`

✏️ **练习 8**：补全下面的函数，跑出估计序列并画图。试试 `g = 0.1, 0.5, 0.9` 有什么不同？

In [ ]:
def simple_filter(measurements, x0, g):
    estimates = []
    x = x0
    for z in measurements:
        prediction = x                 # 1. 预测
        # TODO: 2. 更新 —— 用 g 把 prediction 往 z 方向挪
        x = prediction                 # <- 改这一行
        estimates.append(x)
    return np.array(estimates)

rng = np.random.default_rng(0)
zs = 160 + rng.normal(0, 3, size=30)

plt.figure(figsize=(7, 3.5))
plt.scatter(range(30), zs, color="tab:red", s=15, label="measurements")
for g in [0.1, 0.5, 0.9]:
    plt.plot(simple_filter(zs, x0=155, g=g), label=f"g={g}")
plt.axhline(160, color="k", ls="--", lw=1)
plt.legend(); plt.show()

<details><summary>参考答案 & 思考</summary>

```python
x = prediction + g * (z - prediction)
```

- `g` 小：很“固执”，平滑但反应慢（初值 155 要很久才追上 160）。
- `g` 大：很“轻信”，紧跟测量，噪声几乎没滤掉。
- 卡尔曼滤波本质上就是在**每一步自动算出最优的 g**（叫卡尔曼增益 K），依据是“预测有多不确定 vs 测量有多不确定”。

这就是书第 1 章要展开讲的内容。
</details>

## 9. 自检清单 ✅

能不看答案回答这些，就可以开始读第 1 章了：

- [ ] `a[1:3]` 取的是哪几个元素？
- [ ] `A * B` 和 `A @ B` 有什么区别？
- [ ] `range(5)` 产生哪些数？
- [ ] 怎么生成可复现的高斯噪声？
- [ ] 练习 8 里 `g` 变大/变小，估计曲线会怎么变？为什么？

📝 **费曼复述**：读完后，在 `my-notes/LEARNING_LOG.md` 里用你自己的话写一段：“我写的 simple_filter 在做什么？g 代表什么？”然后发给我，我帮你指出不准确或遗漏的地方。